# Analog to Digital

This notebook introduces sampling: turning a continuous-time waveform into discrete values at fixed time intervals. It connects sample spacing to what a converter can and cannot represent.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


## Sampling a Sine Wave

Sampling does not store every point on the analog curve. It stores a sequence of values taken every `1/fs` seconds.

In [ ]:
analog_fs = 200_000
signal_freq = 1200
t_analog = np.arange(0, 0.008, 1 / analog_fs)
analog = np.cos(2 * np.pi * signal_freq * t_analog)

fig, ax = plt.subplots(figsize=(10, 3.5))

def update_sampling(fs_sample=10_000):
    ax.clear()
    t_samples = np.arange(0, t_analog[-1], 1 / fs_sample)
    samples = np.cos(2 * np.pi * signal_freq * t_samples)
    ax.plot(t_analog * 1e3, analog, label="Analog reference")
    ax.stem(t_samples * 1e3, samples, linefmt="tab:red", markerfmt="ro", basefmt=" ", label="Samples")
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Amplitude")
    ax.set_title(f"Sampling at {fs_sample:.0f} Hz")
    ax.legend()
    fig.canvas.draw_idle()

controls = widgets.interactive(
    update_sampling,
    fs_sample=float_slider(min_value=2500, max_value=30000, step=500, value=10000, description="fs"),
)
display(controls)


## Staircase Reconstruction

Real DACs reconstruct from discrete samples. A simple zero-order hold makes a staircase; an analog low-pass filter smooths that staircase into something closer to the original.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))

def update_reconstruction(fs_sample=8000):
    ax.clear()
    t_samples = np.arange(0, t_analog[-1], 1 / fs_sample)
    samples = np.cos(2 * np.pi * signal_freq * t_samples)
    ax.plot(t_analog * 1e3, analog, label="Analog reference", alpha=0.5)
    ax.step(t_samples * 1e3, samples, where="post", label="Zero-order hold", color="tab:orange")
    ax.plot(t_samples * 1e3, samples, "o", color="tab:red")
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Amplitude")
    ax.set_title("Staircase Reconstruction")
    ax.legend()
    fig.canvas.draw_idle()

controls = widgets.interactive(
    update_reconstruction,
    fs_sample=float_slider(min_value=2500, max_value=30000, step=500, value=8000, description="fs"),
)
display(controls)


## Key Takeaway

Digital signals are not magic analog copies. They are time-spaced measurements, and everything about fidelity depends on how often you take them and how you reconstruct them later.